# Auditoría PRE-BERT del corpus para regeneración del dataset contextual

## Objetivo

Este notebook se ejecuta **antes de regenerar el dataset contextual**. Su propósito es determinar cuánta cobertura contextual utilizable existe en las mismas fuentes de corpus y bajo las mismas reglas de limpieza/ventana que el generador contextual definitivo.

La unidad fundamental sigue siendo una **ocurrencia contextual**. Este notebook **no genera embeddings BERT** y no reemplaza el dataset por prototipos.

El flujo posterior esperado es:

\[
\text{corpus}
\rightarrow
\text{ocurrencias contextuales}
\rightarrow
\text{BERT raw por ocurrencia}
\rightarrow
\begin{cases}
\text{Exp2\_prototipos: agregación raw por palabra}\\
\text{E6b/E6c: análisis occurrence-level}
\end{cases}
\]

## Invariantes heredadas del generador contextual

Se mantienen:

- tokenización estricta con tokens `[a-z]+`;
- normalización de texto;
- exclusión de stopwords y términos basura como **palabras representadas**, pero no del contexto;
- ventana centrada de ±40 tokens;
- mínimo de 4 tokens por texto/ventana;
- deduplicación conceptual por `(word, context_window)`;
- `doc_split` determinístico por documento fuente;
- mismas 24 palabras Exp2 y sus 8 clases;
- mismas fuentes generales y límites de recorrido;
- fallback dirigido de Wikipedia solo para targets que no alcancen el mínimo;
- máximo auditado de 300 contextos únicos por palabra.

## Qué se audita

1. cantidad de contextos únicos utilizables por palabra, truncada a 300;
2. vocabulario que alcanza ≥20, ≥50, ≥75, ≥100, ≥150, ≥200, ≥250 y ≥300;
3. cobertura de las 24 palabras Exp2;
4. cobertura por fuente y `doc_split`;
5. necesidad de fallback dirigido para targets;
6. tamaño proyectado del dataset para caps de 100–300 ocurrencias por palabra;
7. almacenamiento proyectado usando, cuando existe, el tamaño real por fila del Parquet contextual actual.

## Importante

El umbral científico principal para ser candidato a prototipo es:

\[
n_w \ge 100
\]

Los demás umbrales se calculan únicamente como diagnóstico.

La auditoría usa hashes BLAKE2b de 128 bits para representar la clave `(word, context_window)` sin almacenar todas las ventanas completas en RAM. Las colisiones son despreciables a esta escala y no alteran la definición conceptual de deduplicación.

In [1]:
# ============================================================
# 01. Imports, rutas y configuración de auditoría
# ============================================================

import gc
import hashlib
import json
import math
import random
import re
import sys
import time
import unicodedata

from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Tuple

import numpy as np
import pandas as pd

from IPython.display import display
from tqdm.auto import tqdm

from datasets import load_dataset

try:
    import pyarrow.parquet as pq
except Exception:
    pq = None


# ============================================================
# 1. Rutas
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\jisaa\Workspace\Tesis")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_ROOT = (
    PROJECT_ROOT
    / "Exp2"
    / "Results"
    / "ResultsExp2_prototipos"
)

AUDIT_ROOT = (
    RESULTS_ROOT
    / "00_corpus_audit"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CURRENT_CONTEXTUAL_PARQUET = (
    PROJECT_ROOT
    / "contextual_dataset_definitivo"
    / "contextual_occurrences_clean_mincoverage.parquet"
)


# ============================================================
# 2. Configuración heredada del generador contextual
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

WINDOW_TOKENS_EACH_SIDE = 40

MIN_LINE_WORDS = 4
MIN_ROW_WORD_LEN = 3

DROP_WIKITEXT_HEADINGS = True

REMOVE_STOPWORDS_FROM_CONTEXT_WINDOW = False

MIN_OCCURRENCES_FOR_PROTOTYPE = 100

# El generador actual limita targets/fillers a 300.
# Auditar hasta 300 es suficiente para:
# - decidir elegibilidad >=100;
# - proyectar caps 100...300;
# - no inflar RAM innecesariamente.
AUDIT_MAX_CONTEXTS_PER_WORD = 300

AUDIT_THRESHOLDS = [
    1,
    2,
    5,
    10,
    20,
    50,
    75,
    100,
    125,
    150,
    200,
    250,
    300,
]

PROJECTION_CAPS = [
    100,
    150,
    200,
    250,
    300,
]

RUN_TARGET_FALLBACK = True

# Referencias obtenidas en la auditoría previa de Exp2_prototipos.
# Son solo referencias descriptivas, no criterios de aceptación.
CURRENT_CONTEXTUAL_V100_REFERENCE = 458
GLOVE_VOCAB_REFERENCE = 399_997

print("PROJECT_ROOT:", PROJECT_ROOT)
print("AUDIT_ROOT:", AUDIT_ROOT)
print("MIN_OCCURRENCES_FOR_PROTOTYPE:", MIN_OCCURRENCES_FOR_PROTOTYPE)
print("AUDIT_MAX_CONTEXTS_PER_WORD:", AUDIT_MAX_CONTEXTS_PER_WORD)
print("CURRENT_CONTEXTUAL_PARQUET existe:", CURRENT_CONTEXTUAL_PARQUET.exists())

PROJECT_ROOT: C:\Users\jisaa\Workspace\Tesis
AUDIT_ROOT: C:\Users\jisaa\Workspace\Tesis\Exp2\Results\ResultsExp2_prototipos\00_corpus_audit
MIN_OCCURRENCES_FOR_PROTOTYPE: 100
AUDIT_MAX_CONTEXTS_PER_WORD: 300
CURRENT_CONTEXTUAL_PARQUET existe: True


In [2]:
# ============================================================
# 02. Helpers de persistencia y salida "PARA COPIAR Y PEGAR"
# ============================================================

def json_safe(value):
    if value is None:
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (str, bool, int)):
        return value

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, (float, np.floating)):
        value = float(value)

        if np.isnan(value):
            return None

        if np.isposinf(value):
            return "Infinity"

        if np.isneginf(value):
            return "-Infinity"

        return value

    if isinstance(value, pd.DataFrame):
        return [
            json_safe(row)
            for row in value.to_dict(orient="records")
        ]

    if isinstance(value, pd.Series):
        return json_safe(value.to_dict())

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set, frozenset),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())

    return str(value)


def save_json(payload, path):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(payload),
            handle,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )

    return path


def print_copy_block(title, payload):
    print("\n" + "=" * 90)
    print(f"PARA COPIAR Y PEGAR — {title}")
    print("=" * 90)

    print(
        json.dumps(
            json_safe(payload),
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
    )

    print("=" * 90)
    print(f"FIN — {title}")
    print("=" * 90)


def safe_fraction(numerator, denominator):
    if denominator == 0:
        return None

    return float(
        numerator
        / denominator
    )

In [3]:
# ============================================================
# 03. Palabras objetivo y contrato de clases Exp2
# ============================================================

try:
    from preprocessing import CategoryWords, TargetWords

    category_words = [
        str(word).lower()
        for word
        in CategoryWords.load()
    ]

    target_extra_words = [
        str(word).lower()
        for word
        in TargetWords.load()
    ]

except Exception as exc:
    print(
        "No se pudo importar CategoryWords/TargetWords desde preprocessing.py."
    )
    print(
        "Se utilizará el fallback idéntico al generador contextual."
    )
    print(
        "Error original:",
        repr(exc),
    )

    category_words = [
        "apple",
        "bicycle",
        "bread",
        "canoe",
        "chair",
        "jellyfish",
        "screwdriver",
        "skunk",
    ]

    target_extra_words = [
        "ant",
        "bus",
        "cheese",
        "crab",
        "curtain",
        "frog",
        "olive",
        "parachute",
        "peanut",
        "pear",
        "raccoon",
        "raft",
        "scooter",
        "sling",
        "stool",
        "wrench",
    ]


target_words = sorted(
    set(
        category_words
        + target_extra_words
    )
)

target_set = set(
    target_words
)

category_set = set(
    category_words
)

target_extra_set = set(
    target_extra_words
)


target_roles = {}

for word in target_words:

    if (
        word in category_set
        and word in target_extra_set
    ):
        target_roles[word] = "category_and_target"

    elif word in category_set:
        target_roles[word] = "category"

    else:
        target_roles[word] = "target"


EXP2_CLASS_LABELS = [
    "apple",
    "bicycle",
    "bread",
    "canoe",
    "chair",
    "jellyfish",
    "screwdriver",
    "skunk",
]


EXP2_CLASS_GROUPS = {
    "apple":
        ["apple", "olive", "pear"],

    "bicycle":
        ["bicycle", "parachute", "scooter"],

    "bread":
        ["bread", "cheese", "peanut"],

    "canoe":
        ["canoe", "bus", "raft"],

    "chair":
        ["chair", "curtain", "stool"],

    "jellyfish":
        ["jellyfish", "ant", "crab"],

    "screwdriver":
        ["screwdriver", "sling", "wrench"],

    "skunk":
        ["skunk", "frog", "raccoon"],
}


WORD_TO_TRUE_CLASS = {}
WORD_TO_CLASS_LABEL = {}

for class_id, class_label in enumerate(
    EXP2_CLASS_LABELS
):

    for word in EXP2_CLASS_GROUPS[
        class_label
    ]:

        WORD_TO_TRUE_CLASS[
            word
        ] = int(
            class_id
        )

        WORD_TO_CLASS_LABEL[
            word
        ] = str(
            class_label
        )


missing_class_words = sorted(
    set(target_words)
    - set(WORD_TO_TRUE_CLASS)
)

extra_class_words = sorted(
    set(WORD_TO_TRUE_CLASS)
    - set(target_words)
)


if missing_class_words:
    raise RuntimeError(
        "Hay target_words sin clase Exp2: "
        f"{missing_class_words}"
    )


if extra_class_words:
    raise RuntimeError(
        "EXP2_CLASS_GROUPS contiene palabras no incluidas "
        f"en target_words: {extra_class_words}"
    )


if len(target_words) != 24:
    raise RuntimeError(
        f"Se esperaban 24 palabras Exp2 y se obtuvieron {len(target_words)}."
    )


if len(category_words) != 8:
    raise RuntimeError(
        f"Se esperaban 8 CategoryWords y se obtuvieron {len(category_words)}."
    )


target_contract_df = pd.DataFrame(
    [
        {
            "word":
                word,

            "word_role":
                target_roles[
                    word
                ],

            "true_class":
                WORD_TO_TRUE_CLASS[
                    word
                ],

            "class_label":
                WORD_TO_CLASS_LABEL[
                    word
                ],

            "is_category_word":
                bool(
                    word
                    in category_set
                ),
        }
        for word
        in target_words
    ]
).sort_values(
    [
        "true_class",
        "word",
    ]
).reset_index(
    drop=True
)


TARGET_CONTRACT_PATH = (
    AUDIT_ROOT
    / "target_words_contract.csv"
)

target_contract_df.to_csv(
    TARGET_CONTRACT_PATH,
    index=False,
    encoding="utf-8",
)


print(
    "N CategoryWords:",
    len(category_words),
)

print(
    "N target extra:",
    len(target_extra_words),
)

print(
    "N total target words:",
    len(target_words),
)

display(
    target_contract_df
)

N CategoryWords: 8
N target extra: 16
N total target words: 24


,word,word_role,true_class,class_label,is_category_word
0,apple,category,0,apple,True
1,olive,target,0,apple,False
2,pear,target,0,apple,False
3,bicycle,category,1,bicycle,True
4,parachute,target,1,bicycle,False
5,scooter,target,1,bicycle,False
6,bread,category,2,bread,True
7,cheese,target,2,bread,False
8,peanut,target,2,bread,False
9,bus,target,3,canoe,False


In [4]:
# ============================================================
# 04. Limpieza y definición de ocurrencia
#     Réplica PRE-BERT del generador contextual
# ============================================================

STRICT_WORD_PATTERN = re.compile(
    r"[a-z]+"
)


try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

    STOPWORDS = set(
        ENGLISH_STOP_WORDS
    )

except Exception:

    STOPWORDS = {
        "a", "an", "the", "and", "or", "but", "if", "while",
        "of", "to", "in", "on", "for", "with", "as", "by",
        "at", "from", "is", "are", "was", "were", "be", "been",
        "being", "this", "that", "these", "those", "it", "its",
        "he", "she", "they", "them", "we", "you", "i", "me",
        "my", "your", "his", "her", "their",
    }


CUSTOM_BAD_WORDS = {
    "nbsp", "amp", "lt", "gt", "ref", "br", "html",
    "http", "https", "www", "com", "org", "net",
    "url", "jpg", "jpeg", "png", "gif", "svg", "px",
    "thumb", "thumbnail", "image", "file", "align",
    "isbn", "utc", "doi", "redirect", "category",
    "template", "cite", "citation", "edit",
}


BAD_WORDS = (
    set(STOPWORDS)
    | set(CUSTOM_BAD_WORDS)
)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(text),
    )

    text = text.lower()

    text = (
        text
        .replace(
            " @-@ ",
            "-",
        )
        .replace(
            " @.@ ",
            ".",
        )
        .replace(
            " @,@ ",
            ",",
        )
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def strict_tokenize(text: str) -> List[str]:
    text = normalize_text(
        text
    )

    return STRICT_WORD_PATTERN.findall(
        text
    )


def is_heading_line(text: str) -> bool:
    return str(
        text
    ).lstrip().startswith(
        "="
    )


def is_valid_row_word(
    word: str,
    *,
    keep_if_target: bool = True,
) -> bool:

    word = str(
        word
    ).lower()

    if (
        keep_if_target
        and word in target_set
    ):
        return True

    if len(
        word
    ) < MIN_ROW_WORD_LEN:
        return False

    if not re.fullmatch(
        r"[a-z]+",
        word,
    ):
        return False

    if word in BAD_WORDS:
        return False

    return True


def stable_doc_split(
    doc_key: str,
    train_p: float = 0.70,
    val_p: float = 0.15,
) -> str:

    h = hashlib.md5(
        doc_key.encode(
            "utf-8"
        )
    ).hexdigest()

    u = (
        int(
            h[:8],
            16,
        )
        / 0xFFFFFFFF
    )

    if u < train_p:
        return "train"

    if u < (
        train_p
        + val_p
    ):
        return "validation"

    return "test"


def centered_window_tokens(
    tokens: List[str],
    word_pos: int,
) -> Tuple[List[str], int]:

    left = max(
        0,
        word_pos
        - WINDOW_TOKENS_EACH_SIDE,
    )

    right = min(
        len(tokens),
        word_pos
        + WINDOW_TOKENS_EACH_SIDE
        + 1,
    )

    window_tokens = tokens[
        left:right
    ]

    word_pos_window = (
        word_pos
        - left
    )

    # El generador definitivo conserva stopwords en el contexto.
    if REMOVE_STOPWORDS_FROM_CONTEXT_WINDOW:
        cleaned = [
            token
            for token
            in window_tokens
            if (
                re.fullmatch(
                    r"[a-z]+",
                    token,
                )
                and (
                    token not in BAD_WORDS
                    or token in target_set
                )
            )
        ]

        word = tokens[
            word_pos
        ]

        positions = [
            idx
            for idx, token
            in enumerate(
                cleaned
            )
            if token == word
        ]

        if not positions:
            return [], -1

        window_tokens = cleaned

        word_pos_window = min(
            positions,
            key=lambda idx:
                abs(
                    idx
                    - len(
                        window_tokens
                    )
                    // 2
                ),
        )

    return (
        window_tokens,
        word_pos_window,
    )


def word_context_digest(
    word: str,
    window_tokens: List[str],
) -> bytes:
    """
    Representación compacta de la clave conceptual:
        (word, context_window)

    El generador usa context_window = " ".join(window_tokens).
    """

    context_window = " ".join(
        window_tokens
    )

    payload = (
        str(word)
        + "\0"
        + context_window
    ).encode(
        "utf-8"
    )

    return hashlib.blake2b(
        payload,
        digest_size=16,
    ).digest()


print(
    "Stopwords:",
    len(STOPWORDS),
)

print(
    "BAD_WORDS:",
    len(BAD_WORDS),
)

print(
    "Ventana máxima:",
    2 * WINDOW_TOKENS_EACH_SIDE + 1,
    "tokens",
)

Stopwords: 318
BAD_WORDS: 352
Ventana máxima: 81 tokens


In [5]:
# ============================================================
# 05. Fuentes de corpus
#     Réplica de contextualized_df
# ============================================================

CORPUS_SOURCES = [
    {
        "source_name":
            "wikitext_103_raw",

        "dataset_name":
            "wikitext",

        "config":
            "wikitext-103-raw-v1",

        "splits":
            [
                "train",
                "validation",
                "test",
            ],

        "text_field":
            "text",

        "streaming":
            False,

        "max_records_per_split":
            None,
    },

    {
        "source_name":
            "wikipedia_20220301_en",

        "dataset_name":
            "wikipedia",

        "config":
            "20220301.en",

        "splits":
            [
                "train",
            ],

        "text_field":
            "text",

        "streaming":
            True,

        "max_records_per_split":
            800_000,
    },

    {
        "source_name":
            "bookcorpusopen",

        "dataset_name":
            "bookcorpusopen",

        "config":
            None,

        "splits":
            [
                "train",
            ],

        "text_field":
            "text",

        "streaming":
            True,

        "max_records_per_split":
            500_000,
    },
]


LOCAL_TEXT_FILES = [
    # r"C:\ruta\a\corpus_extra.txt",
]


# Fallback dirigido idéntico al generador actual.
FALLBACK_DATASET_NAME = "wikimedia/wikipedia"
FALLBACK_DATASET_CONFIG = "20231101.en"
FALLBACK_SPLIT = "train"
FALLBACK_MAX_STREAM_EXAMPLES = 3_000_000


def load_hf_split(
    dataset_name: str,
    config: Optional[str],
    split: str,
    streaming: bool,
):

    if config is None:

        return load_dataset(
            dataset_name,
            split=split,
            streaming=streaming,
        )

    return load_dataset(
        dataset_name,
        config,
        split=split,
        streaming=streaming,
    )


def iter_configured_source_splits():
    """
    Produce descriptores de source/split.

    Los errores de carga se registrarán durante la auditoría
    y no se ocultarán.
    """

    for source in CORPUS_SOURCES:

        for split in source[
            "splits"
        ]:

            yield {
                **source,
                "split":
                    split,
            }


print(
    "Fuentes generales configuradas:",
    len(CORPUS_SOURCES),
)

display(
    pd.DataFrame(
        CORPUS_SOURCES
    )
)

Fuentes generales configuradas: 3


,source_name,dataset_name,config,splits,text_field,streaming,max_records_per_split
0,wikitext_103_raw,wikitext,wikitext-103-raw-v1,"[train, validation, test]",text,False,NaN
1,wikipedia_20220301_en,wikipedia,20220301.en,[train],text,True,800000.0
2,bookcorpusopen,bookcorpusopen,NaN,[train],text,True,500000.0


## Contrato de estructura del dataset regenerado

Esta auditoría **no cambia la estructura occurrence-level** del dataset definitivo.

La regeneración posterior debe mantener una fila por ocurrencia y conservar el contrato:

`occ_id`, `candidate_id`, `word`, `is_target`, `word_role`, `true_class`, `class_label`, `is_exp2_word`, `selection_stage`, `source_name`, `source_split`, `source_row_idx`, `source_doc_key`, `doc_split`, `word_pos_original`, `word_pos_window`, `wp_start`, `wp_end`, `n_tokens_window`, `context_window`, `left_context`, `right_context`, `text`, `embedding`.

El prototipo **no pertenece a este esquema base**. En `Exp2_prototipos.ipynb` se deriva después mediante:

\[
p_w=
\frac{1}{n_w}
\sum_i x_{w,i}^{raw}.
\]

In [6]:
# ============================================================
# 06. Guardar contrato de compatibilidad occurrence-level
# ============================================================

OCCURRENCE_SCHEMA_CONTRACT = [
    "occ_id",
    "candidate_id",
    "word",
    "is_target",
    "word_role",
    "true_class",
    "class_label",
    "is_exp2_word",
    "selection_stage",
    "source_name",
    "source_split",
    "source_row_idx",
    "source_doc_key",
    "doc_split",
    "word_pos_original",
    "word_pos_window",
    "wp_start",
    "wp_end",
    "n_tokens_window",
    "context_window",
    "left_context",
    "right_context",
    "text",
    "embedding",
]


schema_contract = {
    "base_unit":
        "contextual_occurrence",

    "prototype_is_derived_not_stored_as_base_unit":
        True,

    "embedding_model_after_audit":
        "bert-base-uncased",

    "embedding_dim_after_audit":
        768,

    "embedding_representation_after_audit":
        "raw_mean_last_4_layers_per_occurrence",

    "l2_normalization_in_base_dataset":
        False,

    "columns":
        OCCURRENCE_SCHEMA_CONTRACT,
}


save_json(
    schema_contract,
    AUDIT_ROOT
    / "occurrence_schema_contract.json",
)

print_copy_block(
    "CONTRATO OCCURRENCE-LEVEL",
    schema_contract,
)


PARA COPIAR Y PEGAR — CONTRATO OCCURRENCE-LEVEL
{
  "base_unit": "contextual_occurrence",
  "columns": [
    "occ_id",
    "candidate_id",
    "word",
    "is_target",
    "word_role",
    "true_class",
    "class_label",
    "is_exp2_word",
    "selection_stage",
    "source_name",
    "source_split",
    "source_row_idx",
    "source_doc_key",
    "doc_split",
    "word_pos_original",
    "word_pos_window",
    "wp_start",
    "wp_end",
    "n_tokens_window",
    "context_window",
    "left_context",
    "right_context",
    "text",
    "embedding"
  ],
  "embedding_dim_after_audit": 768,
  "embedding_model_after_audit": "bert-base-uncased",
  "embedding_representation_after_audit": "raw_mean_last_4_layers_per_occurrence",
  "l2_normalization_in_base_dataset": false,
  "prototype_is_derived_not_stored_as_base_unit": true
}
FIN — CONTRATO OCCURRENCE-LEVEL


In [7]:
# ============================================================
# 07. Motor de auditoría general PRE-BERT
# ============================================================
#
# Estrategia:
#
# - recorre las mismas fuentes;
# - aplica la misma tokenización y filtros;
# - para cada palabra válida conserva como máximo 300
#   contextos únicos;
# - deduplica conceptualmente por (word, context_window);
# - NO guarda embeddings;
# - NO construye candidates_df completo;
# - NO guarda ventanas completas en RAM.
#
# El set global contiene hashes de 128 bits de las parejas
# (word, context_window), reduciendo sustancialmente memoria.
# ============================================================


def make_empty_audit_state():

    return {
        "seen_pair_hashes":
            set(),

        "raw_valid_word_counts":
            Counter(),

        "usable_context_counts":
            Counter(),

        "accepted_by_source":
            Counter(),

        "accepted_by_source_split":
            Counter(),

        "accepted_by_doc_split":
            Counter(),

        "accepted_by_word_source":
            Counter(),

        "accepted_by_word_doc_split":
            Counter(),

        "source_unique_words":
            defaultdict(
                set
            ),

        "source_split_stats":
            [],

        "source_failures":
            [],

        "global_skipped":
            Counter(),

        "n_documents_seen":
            0,

        "n_tokens_seen":
            0,
    }


def _accept_occurrence_if_new(
    *,
    state,
    word,
    tokens,
    position,
    source_name,
    source_split,
    source_row_idx,
):
    """
    Devuelve True solo si la ocurrencia sería estructuralmente
    utilizable y aporta un nuevo (word, context_window).

    El conteo se trunca en AUDIT_MAX_CONTEXTS_PER_WORD.
    """

    if (
        state[
            "usable_context_counts"
        ][
            word
        ]
        >= AUDIT_MAX_CONTEXTS_PER_WORD
    ):
        state[
            "global_skipped"
        ][
            "word_cap_reached"
        ] += 1

        return False

    (
        window_tokens,
        word_pos_window,
    ) = centered_window_tokens(
        tokens,
        position,
    )

    if (
        not window_tokens
        or word_pos_window < 0
        or word_pos_window >= len(
            window_tokens
        )
    ):
        state[
            "global_skipped"
        ][
            "bad_window"
        ] += 1

        return False

    if (
        window_tokens[
            word_pos_window
        ]
        != word
    ):
        state[
            "global_skipped"
        ][
            "miscentered_window"
        ] += 1

        return False

    if len(
        window_tokens
    ) < MIN_LINE_WORDS:
        state[
            "global_skipped"
        ][
            "window_too_short"
        ] += 1

        return False

    digest = word_context_digest(
        word,
        window_tokens,
    )

    if digest in state[
        "seen_pair_hashes"
    ]:
        state[
            "global_skipped"
        ][
            "duplicate_word_context"
        ] += 1

        return False

    state[
        "seen_pair_hashes"
    ].add(
        digest
    )

    state[
        "usable_context_counts"
    ][
        word
    ] += 1

    state[
        "accepted_by_source"
    ][
        source_name
    ] += 1

    state[
        "accepted_by_source_split"
    ][
        (
            source_name,
            source_split,
        )
    ] += 1

    source_doc_key = (
        f"{source_name}:"
        f"{source_split}:"
        f"{source_row_idx}"
    )

    doc_split = stable_doc_split(
        source_doc_key
    )

    state[
        "accepted_by_doc_split"
    ][
        doc_split
    ] += 1

    state[
        "accepted_by_word_source"
    ][
        (
            word,
            source_name,
        )
    ] += 1

    state[
        "accepted_by_word_doc_split"
    ][
        (
            word,
            doc_split,
        )
    ] += 1

    state[
        "source_unique_words"
    ][
        source_name
    ].add(
        word
    )

    return True


def scan_one_hf_source_split(
    *,
    state,
    source_name,
    dataset_name,
    config,
    split,
    text_field,
    streaming,
    max_records_per_split,
):
    """
    Escanea un source/split completo o hasta su límite configurado.
    """

    started = time.perf_counter()

    stats = Counter()

    print(
        "\n"
        + "=" * 80
    )

    print(
        "AUDITANDO:",
        source_name,
        "| split=",
        split,
        "| streaming=",
        streaming,
        "| max_records=",
        max_records_per_split,
    )

    print(
        "=" * 80
    )

    try:

        dataset = load_hf_split(
            dataset_name,
            config,
            split,
            streaming,
        )

    except Exception as exc:

        failure = {
            "source_name":
                source_name,

            "split":
                split,

            "dataset_name":
                dataset_name,

            "config":
                config,

            "error":
                repr(
                    exc
                ),
        }

        state[
            "source_failures"
        ].append(
            failure
        )

        print(
            "NO SE PUDO CARGAR LA FUENTE:",
            failure,
        )

        return


    total_for_progress = (
        max_records_per_split
        if max_records_per_split
        is not None
        else (
            len(dataset)
            if (
                not streaming
                and hasattr(
                    dataset,
                    "__len__",
                )
            )
            else None
        )
    )


    iterator = tqdm(
        enumerate(
            dataset
        ),
        total=total_for_progress,
        desc=f"{source_name}/{split}",
        unit="docs",
    )


    for row_idx, row in iterator:

        if (
            max_records_per_split
            is not None
            and row_idx
            >= max_records_per_split
        ):
            break

        stats[
            "documents_seen"
        ] += 1

        state[
            "n_documents_seen"
        ] += 1

        text = row.get(
            text_field,
            "",
        )

        if (
            text is None
            or not isinstance(
                text,
                str,
            )
            or not text.strip()
        ):
            stats[
                "empty_text"
            ] += 1

            continue

        if (
            DROP_WIKITEXT_HEADINGS
            and is_heading_line(
                text
            )
        ):
            stats[
                "heading"
            ] += 1

            continue

        tokens = strict_tokenize(
            text
        )

        stats[
            "tokens_seen"
        ] += len(
            tokens
        )

        state[
            "n_tokens_seen"
        ] += len(
            tokens
        )

        if len(
            tokens
        ) < MIN_LINE_WORDS:
            stats[
                "too_short"
            ] += 1

            continue

        stats[
            "documents_usable"
        ] += 1

        accepted_this_document = 0

        for position, word in enumerate(
            tokens
        ):

            if not is_valid_row_word(
                word,
                keep_if_target=True,
            ):
                continue

            state[
                "raw_valid_word_counts"
            ][
                word
            ] += 1

            stats[
                "valid_word_tokens"
            ] += 1

            accepted = _accept_occurrence_if_new(
                state=state,
                word=word,
                tokens=tokens,
                position=position,
                source_name=source_name,
                source_split=split,
                source_row_idx=int(
                    row_idx
                ),
            )

            if accepted:
                stats[
                    "unique_contexts_accepted"
                ] += 1

                accepted_this_document += 1

        if accepted_this_document > 0:
            stats[
                "documents_contributing"
            ] += 1

        if (
            row_idx > 0
            and row_idx % 5000 == 0
        ):

            n_v100_now = sum(
                count
                >= MIN_OCCURRENCES_FOR_PROTOTYPE
                for count
                in state[
                    "usable_context_counts"
                ].values()
            )

            iterator.set_postfix(
                {
                    "V100":
                        int(
                            n_v100_now
                        ),

                    "unique_ctx":
                        int(
                            len(
                                state[
                                    "seen_pair_hashes"
                                ]
                            )
                        ),
                }
            )


    elapsed_seconds = float(
        time.perf_counter()
        - started
    )


    stats_row = {
        "source_name":
            source_name,

        "split":
            split,

        "dataset_name":
            dataset_name,

        "config":
            config,

        "streaming":
            bool(
                streaming
            ),

        "max_records_per_split":
            max_records_per_split,

        "elapsed_seconds":
            elapsed_seconds,

        **{
            str(key):
                int(
                    value
                )
            for key, value
            in stats.items()
        },
    }


    state[
        "source_split_stats"
    ].append(
        stats_row
    )


    print(
        "\nResumen source/split:"
    )

    print(
        json.dumps(
            json_safe(
                stats_row
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    # Guardar progreso después de cada source/split.
    pd.DataFrame(
        state[
            "source_split_stats"
        ]
    ).to_csv(
        AUDIT_ROOT
        / "source_split_progress.csv",
        index=False,
        encoding="utf-8",
    )

    gc.collect()


def scan_local_text_files(
    state,
):

    for file_path in LOCAL_TEXT_FILES:

        file_path = Path(
            file_path
        )

        if not file_path.exists():

            state[
                "source_failures"
            ].append(
                {
                    "source_name":
                        f"local_file:{file_path.name}",

                    "split":
                        "local",

                    "dataset_name":
                        None,

                    "config":
                        None,

                    "error":
                        "file_not_found",
                }
            )

            continue

        source_name = (
            f"local_file:{file_path.name}"
        )

        started = time.perf_counter()

        stats = Counter()

        with open(
            file_path,
            "r",
            encoding="utf-8",
            errors="ignore",
        ) as handle:

            iterator = tqdm(
                enumerate(
                    handle
                ),
                desc=source_name,
                unit="docs",
            )

            for row_idx, text in iterator:

                text = str(
                    text
                ).strip()

                if not text:
                    continue

                stats[
                    "documents_seen"
                ] += 1

                state[
                    "n_documents_seen"
                ] += 1

                tokens = strict_tokenize(
                    text
                )

                stats[
                    "tokens_seen"
                ] += len(
                    tokens
                )

                state[
                    "n_tokens_seen"
                ] += len(
                    tokens
                )

                if len(
                    tokens
                ) < MIN_LINE_WORDS:
                    stats[
                        "too_short"
                    ] += 1

                    continue

                stats[
                    "documents_usable"
                ] += 1

                for position, word in enumerate(
                    tokens
                ):

                    if not is_valid_row_word(
                        word,
                        keep_if_target=True,
                    ):
                        continue

                    state[
                        "raw_valid_word_counts"
                    ][
                        word
                    ] += 1

                    stats[
                        "valid_word_tokens"
                    ] += 1

                    accepted = _accept_occurrence_if_new(
                        state=state,
                        word=word,
                        tokens=tokens,
                        position=position,
                        source_name=source_name,
                        source_split="local",
                        source_row_idx=int(
                            row_idx
                        ),
                    )

                    if accepted:
                        stats[
                            "unique_contexts_accepted"
                        ] += 1


        state[
            "source_split_stats"
        ].append(
            {
                "source_name":
                    source_name,

                "split":
                    "local",

                "dataset_name":
                    None,

                "config":
                    None,

                "streaming":
                    False,

                "max_records_per_split":
                    None,

                "elapsed_seconds":
                    float(
                        time.perf_counter()
                        - started
                    ),

                **{
                    str(key):
                        int(
                            value
                        )
                    for key, value
                    in stats.items()
                },
            }
        )


def run_general_corpus_audit():

    state = make_empty_audit_state()

    started = time.perf_counter()

    for source_split in iter_configured_source_splits():

        scan_one_hf_source_split(
            state=state,
            source_name=source_split[
                "source_name"
            ],
            dataset_name=source_split[
                "dataset_name"
            ],
            config=source_split[
                "config"
            ],
            split=source_split[
                "split"
            ],
            text_field=source_split[
                "text_field"
            ],
            streaming=bool(
                source_split[
                    "streaming"
                ]
            ),
            max_records_per_split=source_split[
                "max_records_per_split"
            ],
        )

    scan_local_text_files(
        state
    )

    state[
        "general_elapsed_seconds"
    ] = float(
        time.perf_counter()
        - started
    )

    return state

In [8]:
# ============================================================
# 08. EJECUCIÓN PESADA — auditar corpus general
# ============================================================
#
# Esta es la celda costosa del notebook.
#
# NO ejecuta BERT.
# NO crea embeddings.
#
# Recorre las fuentes configuradas y cuenta contextos únicos
# utilizables, truncados a 300 por palabra.
# ============================================================

general_state = (
    run_general_corpus_audit()
)

print(
    "\nAuditoría general terminada."
)

print(
    "Documentos recorridos:",
    general_state[
        "n_documents_seen"
    ],
)

print(
    "Tokens recorridos:",
    general_state[
        "n_tokens_seen"
    ],
)

print(
    "Pares únicos (word, context) retenidos:",
    len(
        general_state[
            "seen_pair_hashes"
        ]
    ),
)

print(
    "Tiempo total [min]:",
    round(
        general_state[
            "general_elapsed_seconds"
        ]
        / 60.0,
        2,
    ),
)


AUDITANDO: wikitext_103_raw | split= train | streaming= False | max_records= None


wikitext_103_raw/train:   0%|          | 0/1801350 [00:00<?, ?docs/s]


Resumen source/split:
{
  "source_name": "wikitext_103_raw",
  "split": "train",
  "dataset_name": "wikitext",
  "config": "wikitext-103-raw-v1",
  "streaming": false,
  "max_records_per_split": null,
  "elapsed_seconds": 211.9296698999824,
  "documents_seen": 1801350,
  "empty_text": 636321,
  "heading": 305497,
  "tokens_seen": 82994332,
  "documents_usable": 823809,
  "valid_word_tokens": 43198973,
  "unique_contexts_accepted": 10474157,
  "documents_contributing": 778272,
  "too_short": 35723
}

AUDITANDO: wikitext_103_raw | split= validation | streaming= False | max_records= None


wikitext_103_raw/validation:   0%|          | 0/3760 [00:00<?, ?docs/s]


Resumen source/split:
{
  "source_name": "wikitext_103_raw",
  "split": "validation",
  "dataset_name": "wikitext",
  "config": "wikitext-103-raw-v1",
  "streaming": false,
  "max_records_per_split": null,
  "elapsed_seconds": 2.4721174000005703,
  "documents_seen": 3760,
  "empty_text": 1299,
  "heading": 620,
  "tokens_seen": 175628,
  "documents_usable": 1779,
  "valid_word_tokens": 91799,
  "unique_contexts_accepted": 11554,
  "documents_contributing": 1657,
  "too_short": 62
}

AUDITANDO: wikitext_103_raw | split= test | streaming= False | max_records= None


wikitext_103_raw/test:   0%|          | 0/4358 [00:00<?, ?docs/s]


Resumen source/split:
{
  "source_name": "wikitext_103_raw",
  "split": "test",
  "dataset_name": "wikitext",
  "config": "wikitext-103-raw-v1",
  "streaming": false,
  "max_records_per_split": null,
  "elapsed_seconds": 2.0501059999805875,
  "documents_seen": 4358,
  "empty_text": 1467,
  "heading": 708,
  "tokens_seen": 197341,
  "documents_usable": 2006,
  "valid_word_tokens": 102195,
  "unique_contexts_accepted": 13946,
  "documents_contributing": 1788,
  "too_short": 177
}

AUDITANDO: wikipedia_20220301_en | split= train | streaming= True | max_records= 800000
NO SE PUDO CARGAR LA FUENTE: {'source_name': 'wikipedia_20220301_en', 'split': 'train', 'dataset_name': 'wikipedia', 'config': '20220301.en', 'error': "RuntimeError('Dataset scripts are no longer supported, but found wikipedia.py')"}

AUDITANDO: bookcorpusopen | split= train | streaming= True | max_records= 500000
NO SE PUDO CARGAR LA FUENTE: {'source_name': 'bookcorpusopen', 'split': 'train', 'dataset_name': 'bookcorpusope

In [9]:
# ============================================================
# 09. Resumen general antes del fallback dirigido
# ============================================================


def build_word_coverage_dataframe(
    state,
):

    words = sorted(
        set(
            state[
                "raw_valid_word_counts"
            ]
        )
        | set(
            state[
                "usable_context_counts"
            ]
        )
    )

    rows = []

    for word in words:

        rows.append(
            {
                "word":
                    word,

                "is_target":
                    bool(
                        word
                        in target_set
                    ),

                "word_role":
                    target_roles.get(
                        word,
                        "filler",
                    ),

                "raw_valid_occurrences_seen":
                    int(
                        state[
                            "raw_valid_word_counts"
                        ][
                            word
                        ]
                    ),

                "n_usable_unique_contexts_capped":
                    int(
                        state[
                            "usable_context_counts"
                        ][
                            word
                        ]
                    ),

                "reached_audit_cap":
                    bool(
                        state[
                            "usable_context_counts"
                        ][
                            word
                        ]
                        >= AUDIT_MAX_CONTEXTS_PER_WORD
                    ),
            }
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "n_usable_unique_contexts_capped",
                "word",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


def build_threshold_dataframe(
    coverage_df,
):

    n_total_words = int(
        len(
            coverage_df
        )
    )

    rows = []

    for threshold in AUDIT_THRESHOLDS:

        n_words = int(
            (
                coverage_df[
                    "n_usable_unique_contexts_capped"
                ]
                >= threshold
            ).sum()
        )

        rows.append(
            {
                "min_occurrences":
                    int(
                        threshold
                    ),

                "n_words":
                    n_words,

                "fraction_of_valid_vocabulary":
                    safe_fraction(
                        n_words,
                        n_total_words,
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


general_word_coverage_df = (
    build_word_coverage_dataframe(
        general_state
    )
)


general_threshold_df = (
    build_threshold_dataframe(
        general_word_coverage_df
    )
)


general_target_coverage_df = (
    pd.DataFrame(
        {
            "word":
                target_words
        }
    )
    .merge(
        general_word_coverage_df[
            [
                "word",
                "raw_valid_occurrences_seen",
                "n_usable_unique_contexts_capped",
            ]
        ],
        on="word",
        how="left",
    )
)


general_target_coverage_df[
    [
        "raw_valid_occurrences_seen",
        "n_usable_unique_contexts_capped",
    ]
] = (
    general_target_coverage_df[
        [
            "raw_valid_occurrences_seen",
            "n_usable_unique_contexts_capped",
        ]
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


general_target_coverage_df[
    "true_class"
] = (
    general_target_coverage_df[
        "word"
    ].map(
        WORD_TO_TRUE_CLASS
    )
)


general_target_coverage_df[
    "class_label"
] = (
    general_target_coverage_df[
        "word"
    ].map(
        WORD_TO_CLASS_LABEL
    )
)


general_target_coverage_df[
    "reaches_min100"
] = (
    general_target_coverage_df[
        "n_usable_unique_contexts_capped"
    ]
    >= MIN_OCCURRENCES_FOR_PROTOTYPE
)


general_target_coverage_df = (
    general_target_coverage_df
    .sort_values(
        [
            "true_class",
            "word",
        ]
    )
    .reset_index(
        drop=True
    )
)


general_source_split_stats_df = pd.DataFrame(
    general_state[
        "source_split_stats"
    ]
)


general_source_rows = []

for source_name, n_contexts in general_state[
    "accepted_by_source"
].items():

    general_source_rows.append(
        {
            "source_name":
                source_name,

            "n_unique_contexts_accepted":
                int(
                    n_contexts
                ),

            "n_unique_words_represented":
                int(
                    len(
                        general_state[
                            "source_unique_words"
                        ][
                            source_name
                        ]
                    )
                ),
        }
    )


general_source_coverage_df = (
    pd.DataFrame(
        general_source_rows
    )
    .sort_values(
        "n_unique_contexts_accepted",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
    if general_source_rows
    else pd.DataFrame(
        columns=[
            "source_name",
            "n_unique_contexts_accepted",
            "n_unique_words_represented",
        ]
    )
)


general_doc_split_df = pd.DataFrame(
    [
        {
            "doc_split":
                split,

            "n_unique_contexts_accepted":
                int(
                    count
                ),
        }
        for split, count
        in general_state[
            "accepted_by_doc_split"
        ].items()
    ]
).sort_values(
    "doc_split"
).reset_index(
    drop=True
)


general_v100 = int(
    (
        general_word_coverage_df[
            "n_usable_unique_contexts_capped"
        ]
        >= MIN_OCCURRENCES_FOR_PROTOTYPE
    ).sum()
)


general_audit_summary = {
    "stage":
        "general_corpus_before_target_fallback",

    "audit_max_contexts_per_word":
        AUDIT_MAX_CONTEXTS_PER_WORD,

    "min_occurrences_for_prototype":
        MIN_OCCURRENCES_FOR_PROTOTYPE,

    "n_documents_seen":
        int(
            general_state[
                "n_documents_seen"
            ]
        ),

    "n_tokens_seen":
        int(
            general_state[
                "n_tokens_seen"
            ]
        ),

    "n_valid_unique_words_observed":
        int(
            len(
                general_word_coverage_df
            )
        ),

    "n_words_ge_100":
        general_v100,

    "v100_gain_vs_current_dataset":
        (
            int(
                general_v100
                - CURRENT_CONTEXTUAL_V100_REFERENCE
            )
        ),

    "v100_ratio_vs_current_dataset":
        safe_fraction(
            general_v100,
            CURRENT_CONTEXTUAL_V100_REFERENCE,
        ),

    "v100_fraction_of_glove_reference":
        safe_fraction(
            general_v100,
            GLOVE_VOCAB_REFERENCE,
        ),

    "n_targets_ge_100":
        int(
            general_target_coverage_df[
                "reaches_min100"
            ].sum()
        ),

    "n_targets_total":
        int(
            len(
                target_words
            )
        ),

    "targets_below_100":
        general_target_coverage_df.loc[
            ~general_target_coverage_df[
                "reaches_min100"
            ],
            "word",
        ].tolist(),

    "n_source_failures":
        int(
            len(
                general_state[
                    "source_failures"
                ]
            )
        ),

    "source_failures":
        general_state[
            "source_failures"
        ],

    "coverage_by_threshold":
        general_threshold_df,

    "target_coverage":
        general_target_coverage_df,

    "source_coverage":
        general_source_coverage_df,

    "doc_split_coverage":
        general_doc_split_df,

    "elapsed_seconds":
        float(
            general_state[
                "general_elapsed_seconds"
            ]
        ),
}


# ============================================================
# Guardar
# ============================================================

general_word_coverage_df.to_csv(
    AUDIT_ROOT
    / "general_word_coverage_capped300.csv",
    index=False,
    encoding="utf-8",
)


general_threshold_df.to_csv(
    AUDIT_ROOT
    / "general_coverage_by_threshold.csv",
    index=False,
    encoding="utf-8",
)


general_target_coverage_df.to_csv(
    AUDIT_ROOT
    / "general_target_coverage.csv",
    index=False,
    encoding="utf-8",
)


general_source_split_stats_df.to_csv(
    AUDIT_ROOT
    / "general_source_split_stats.csv",
    index=False,
    encoding="utf-8",
)


general_source_coverage_df.to_csv(
    AUDIT_ROOT
    / "general_source_coverage.csv",
    index=False,
    encoding="utf-8",
)


general_doc_split_df.to_csv(
    AUDIT_ROOT
    / "general_doc_split_coverage.csv",
    index=False,
    encoding="utf-8",
)


save_json(
    general_state[
        "source_failures"
    ],
    AUDIT_ROOT
    / "general_source_failures.json",
)


save_json(
    general_audit_summary,
    AUDIT_ROOT
    / "general_corpus_audit_summary.json",
)


# ============================================================
# Mostrar
# ============================================================

print(
    "\n=== Cobertura por mínimo ==="
)

display(
    general_threshold_df
)


print(
    "\n=== Cobertura 24 targets ==="
)

display(
    general_target_coverage_df
)


print(
    "\n=== Cobertura por fuente ==="
)

display(
    general_source_coverage_df
)


print(
    "\n=== Cobertura por doc_split ==="
)

display(
    general_doc_split_df
)


print_copy_block(
    "AUDITORÍA GENERAL PRE-BERT",
    general_audit_summary,
)


=== Cobertura por mínimo ===


,min_occurrences,n_words,fraction_of_valid_vocabulary
0,1,469330,1.000000
1,2,279127,0.594735
2,5,159067,0.338924
3,10,108721,0.231652
4,20,74965,0.159728
5,50,45715,0.097405
6,75,36216,0.077165
7,100,30567,0.065129
8,125,26768,0.057034
9,150,23971,0.051075



=== Cobertura 24 targets ===


,word,raw_valid_occurrences_seen,n_usable_unique_contexts_capped,true_class,class_label,reaches_min100
0,apple,2627,300,0,apple,True
1,olive,942,300,0,apple,True
2,pear,175,173,0,apple,True
3,bicycle,780,300,1,bicycle,True
4,parachute,1812,300,1,bicycle,True
5,scooter,49,49,1,bicycle,False
6,bread,941,300,2,bread,True
7,cheese,812,300,2,bread,True
8,peanut,101,101,2,bread,True
9,bus,4315,300,3,canoe,True



=== Cobertura por fuente ===


,source_name,n_unique_contexts_accepted,n_unique_words_represented
0,wikitext_103_raw,10499657,469330



=== Cobertura por doc_split ===


,doc_split,n_unique_contexts_accepted
0,test,1573033
1,train,7346842
2,validation,1579782



PARA COPIAR Y PEGAR — AUDITORÍA GENERAL PRE-BERT
{
  "audit_max_contexts_per_word": 300,
  "coverage_by_threshold": [
    {
      "fraction_of_valid_vocabulary": 1.0,
      "min_occurrences": 1,
      "n_words": 469330
    },
    {
      "fraction_of_valid_vocabulary": 0.5947350478341465,
      "min_occurrences": 2,
      "n_words": 279127
    },
    {
      "fraction_of_valid_vocabulary": 0.3389235719003686,
      "min_occurrences": 5,
      "n_words": 159067
    },
    {
      "fraction_of_valid_vocabulary": 0.23165150320669892,
      "min_occurrences": 10,
      "n_words": 108721
    },
    {
      "fraction_of_valid_vocabulary": 0.15972769692966568,
      "min_occurrences": 20,
      "n_words": 74965
    },
    {
      "fraction_of_valid_vocabulary": 0.0974048111137153,
      "min_occurrences": 50,
      "n_words": 45715
    },
    {
      "fraction_of_valid_vocabulary": 0.07716532077642597,
      "min_occurrences": 75,
      "n_words": 36216
    },
    {
      "fraction_of_valid_

In [10]:
# ============================================================
# 10. Fallback dirigido para targets bajo 100
# ============================================================
#
# Replica la lógica del generador actual:
# solo busca más contextos para las palabras Exp2 que aún
# no alcanzan MIN_OCCURRENCES_FOR_PROTOTYPE.
# ============================================================


def run_target_fallback(
    state,
):

    before_counts = {
        word:
            int(
                state[
                    "usable_context_counts"
                ][
                    word
                ]
            )
        for word
        in target_words
    }


    missing_before = [
        word
        for word
        in target_words
        if before_counts[
            word
        ]
        < MIN_OCCURRENCES_FOR_PROTOTYPE
    ]


    report = {
        "fallback_enabled":
            bool(
                RUN_TARGET_FALLBACK
            ),

        "fallback_used":
            False,

        "dataset_name":
            FALLBACK_DATASET_NAME,

        "dataset_config":
            FALLBACK_DATASET_CONFIG,

        "split":
            FALLBACK_SPLIT,

        "max_stream_examples":
            FALLBACK_MAX_STREAM_EXAMPLES,

        "missing_before":
            missing_before,

        "counts_before":
            before_counts,

        "added_by_word":
            {},

        "missing_after":
            missing_before,

        "counts_after":
            before_counts,

        "documents_seen":
            0,

        "elapsed_seconds":
            0.0,

        "error":
            None,
    }


    if (
        not RUN_TARGET_FALLBACK
        or not missing_before
    ):
        return report


    print(
        "\nEjecutando fallback dirigido para:",
        missing_before,
    )


    started = time.perf_counter()

    try:

        dataset = load_dataset(
            FALLBACK_DATASET_NAME,
            FALLBACK_DATASET_CONFIG,
            split=FALLBACK_SPLIT,
            streaming=True,
        )

    except Exception as exc:

        report[
            "error"
        ] = repr(
            exc
        )

        report[
            "elapsed_seconds"
        ] = float(
            time.perf_counter()
            - started
        )

        return report


    report[
        "fallback_used"
    ] = True


    added_by_word = Counter()


    iterator = tqdm(
        enumerate(
            dataset
        ),
        total=FALLBACK_MAX_STREAM_EXAMPLES,
        desc="Target fallback",
        unit="docs",
    )


    for row_idx, row in iterator:

        if row_idx >= FALLBACK_MAX_STREAM_EXAMPLES:
            break


        still_missing = [
            word
            for word
            in target_words
            if state[
                "usable_context_counts"
            ][
                word
            ]
            < MIN_OCCURRENCES_FOR_PROTOTYPE
        ]


        if not still_missing:
            break


        report[
            "documents_seen"
        ] += 1


        text = row.get(
            "text",
            "",
        )


        if (
            not isinstance(
                text,
                str,
            )
            or not text.strip()
        ):
            continue


        if (
            DROP_WIKITEXT_HEADINGS
            and is_heading_line(
                text
            )
        ):
            continue


        tokens = strict_tokenize(
            text
        )


        if len(
            tokens
        ) < MIN_LINE_WORDS:
            continue


        still_missing_set = set(
            still_missing
        )


        if set(
            tokens
        ).isdisjoint(
            still_missing_set
        ):
            continue


        source_name = (
            "wikimedia_wikipedia_20231101_en_fallback"
        )

        source_split = (
            "train_stream"
        )


        for position, word in enumerate(
            tokens
        ):

            if word not in still_missing_set:
                continue


            # raw_valid_word_counts también registra lo observado
            # durante el fallback.
            state[
                "raw_valid_word_counts"
            ][
                word
            ] += 1


            before = int(
                state[
                    "usable_context_counts"
                ][
                    word
                ]
            )


            accepted = _accept_occurrence_if_new(
                state=state,
                word=word,
                tokens=tokens,
                position=position,
                source_name=source_name,
                source_split=source_split,
                source_row_idx=int(
                    row_idx
                ),
            )


            if accepted:

                after = int(
                    state[
                        "usable_context_counts"
                    ][
                        word
                    ]
                )

                added_by_word[
                    word
                ] += (
                    after
                    - before
                )


        if (
            row_idx > 0
            and row_idx % 5000 == 0
        ):

            iterator.set_postfix(
                {
                    "still_missing":
                        len(
                            [
                                word
                                for word
                                in target_words
                                if state[
                                    "usable_context_counts"
                                ][
                                    word
                                ]
                                < MIN_OCCURRENCES_FOR_PROTOTYPE
                            ]
                        ),

                    "added":
                        int(
                            sum(
                                added_by_word.values()
                            )
                        ),
                }
            )


    counts_after = {
        word:
            int(
                state[
                    "usable_context_counts"
                ][
                    word
                ]
            )
        for word
        in target_words
    }


    missing_after = [
        word
        for word
        in target_words
        if counts_after[
            word
        ]
        < MIN_OCCURRENCES_FOR_PROTOTYPE
    ]


    report.update(
        {
            "added_by_word":
                {
                    str(word):
                        int(
                            count
                        )
                    for word, count
                    in added_by_word.items()
                },

            "missing_after":
                missing_after,

            "counts_after":
                counts_after,

            "elapsed_seconds":
                float(
                    time.perf_counter()
                    - started
                ),
        }
    )


    return report


target_fallback_report = (
    run_target_fallback(
        general_state
    )
)


save_json(
    target_fallback_report,
    AUDIT_ROOT
    / "target_fallback_report.json",
)


print_copy_block(
    "FALLBACK DIRIGIDO TARGETS",
    target_fallback_report,
)


Ejecutando fallback dirigido para: ['scooter', 'screwdriver', 'skunk', 'sling', 'wrench']


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Target fallback:   0%|          | 0/3000000 [00:00<?, ?docs/s]


PARA COPIAR Y PEGAR — FALLBACK DIRIGIDO TARGETS
{
  "added_by_word": {
    "scooter": 51,
    "screwdriver": 36,
    "skunk": 21,
    "sling": 37,
    "wrench": 74
  },
  "counts_after": {
    "ant": 300,
    "apple": 300,
    "bicycle": 300,
    "bread": 300,
    "bus": 300,
    "canoe": 300,
    "chair": 300,
    "cheese": 300,
    "crab": 300,
    "curtain": 300,
    "frog": 300,
    "jellyfish": 163,
    "olive": 300,
    "parachute": 300,
    "peanut": 101,
    "pear": 173,
    "raccoon": 300,
    "raft": 289,
    "scooter": 100,
    "screwdriver": 100,
    "skunk": 100,
    "sling": 107,
    "stool": 134,
    "wrench": 120
  },
  "counts_before": {
    "ant": 300,
    "apple": 300,
    "bicycle": 300,
    "bread": 300,
    "bus": 300,
    "canoe": 300,
    "chair": 300,
    "cheese": 300,
    "crab": 300,
    "curtain": 300,
    "frog": 300,
    "jellyfish": 163,
    "olive": 300,
    "parachute": 300,
    "peanut": 101,
    "pear": 173,
    "raccoon": 300,
    "raft": 289,
    

In [11]:
# ============================================================
# 11. Cobertura consolidada después del fallback
# ============================================================

final_word_coverage_df = (
    build_word_coverage_dataframe(
        general_state
    )
)


final_threshold_df = (
    build_threshold_dataframe(
        final_word_coverage_df
    )
)


final_target_coverage_df = (
    pd.DataFrame(
        {
            "word":
                target_words
        }
    )
    .merge(
        final_word_coverage_df[
            [
                "word",
                "raw_valid_occurrences_seen",
                "n_usable_unique_contexts_capped",
            ]
        ],
        on="word",
        how="left",
    )
)


final_target_coverage_df[
    [
        "raw_valid_occurrences_seen",
        "n_usable_unique_contexts_capped",
    ]
] = (
    final_target_coverage_df[
        [
            "raw_valid_occurrences_seen",
            "n_usable_unique_contexts_capped",
        ]
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


final_target_coverage_df[
    "true_class"
] = (
    final_target_coverage_df[
        "word"
    ].map(
        WORD_TO_TRUE_CLASS
    )
)


final_target_coverage_df[
    "class_label"
] = (
    final_target_coverage_df[
        "word"
    ].map(
        WORD_TO_CLASS_LABEL
    )
)


final_target_coverage_df[
    "reaches_min100"
] = (
    final_target_coverage_df[
        "n_usable_unique_contexts_capped"
    ]
    >= MIN_OCCURRENCES_FOR_PROTOTYPE
)


final_target_coverage_df = (
    final_target_coverage_df
    .sort_values(
        [
            "true_class",
            "word",
        ]
    )
    .reset_index(
        drop=True
    )
)


eligible_vocab_df = (
    final_word_coverage_df.loc[
        final_word_coverage_df[
            "n_usable_unique_contexts_capped"
        ]
        >= MIN_OCCURRENCES_FOR_PROTOTYPE
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


eligible_words_final = set(
    eligible_vocab_df[
        "word"
    ]
)


if not set(
    target_words
).issubset(
    eligible_words_final
):
    missing_targets_final = sorted(
        set(
            target_words
        )
        - eligible_words_final
    )
else:
    missing_targets_final = []


# ============================================================
# Composición source/doc_split para vocabulario finalmente elegible
# usando todos los contextos auditados hasta cap=300.
# ============================================================

eligible_source_counter = Counter()

for (
    word,
    source_name,
), count in general_state[
    "accepted_by_word_source"
].items():

    if word in eligible_words_final:

        eligible_source_counter[
            source_name
        ] += int(
            count
        )


eligible_doc_split_counter = Counter()

for (
    word,
    doc_split,
), count in general_state[
    "accepted_by_word_doc_split"
].items():

    if word in eligible_words_final:

        eligible_doc_split_counter[
            doc_split
        ] += int(
            count
        )


eligible_source_df = pd.DataFrame(
    [
        {
            "source_name":
                source,

            "n_contexts_for_eligible_vocabulary_cap300":
                int(
                    count
                ),
        }
        for source, count
        in eligible_source_counter.items()
    ]
).sort_values(
    "n_contexts_for_eligible_vocabulary_cap300",
    ascending=False,
).reset_index(
    drop=True
)


eligible_doc_split_df = pd.DataFrame(
    [
        {
            "doc_split":
                split,

            "n_contexts_for_eligible_vocabulary_cap300":
                int(
                    count
                ),
        }
        for split, count
        in eligible_doc_split_counter.items()
    ]
).sort_values(
    "doc_split"
).reset_index(
    drop=True
)


final_v100 = int(
    len(
        eligible_vocab_df
    )
)


final_summary = {
    "stage":
        "final_after_target_fallback",

    "min_occurrences_for_prototype":
        MIN_OCCURRENCES_FOR_PROTOTYPE,

    "audit_max_contexts_per_word":
        AUDIT_MAX_CONTEXTS_PER_WORD,

    "n_valid_unique_words_observed":
        int(
            len(
                final_word_coverage_df
            )
        ),

    "n_eligible_words_ge_100":
        final_v100,

    "eligible_gain_vs_current_458":
        int(
            final_v100
            - CURRENT_CONTEXTUAL_V100_REFERENCE
        ),

    "eligible_ratio_vs_current_458":
        safe_fraction(
            final_v100,
            CURRENT_CONTEXTUAL_V100_REFERENCE,
        ),

    "eligible_fraction_of_glove_reference":
        safe_fraction(
            final_v100,
            GLOVE_VOCAB_REFERENCE,
        ),

    "n_targets_ge_100":
        int(
            final_target_coverage_df[
                "reaches_min100"
            ].sum()
        ),

    "n_targets_total":
        int(
            len(
                target_words
            )
        ),

    "all_targets_ge_100":
        bool(
            final_target_coverage_df[
                "reaches_min100"
            ].all()
        ),

    "targets_below_100":
        missing_targets_final,

    "coverage_by_threshold":
        final_threshold_df,

    "target_coverage":
        final_target_coverage_df,

    "eligible_vocabulary_source_composition_cap300":
        eligible_source_df,

    "eligible_vocabulary_doc_split_composition_cap300":
        eligible_doc_split_df,
}


# ============================================================
# Guardar
# ============================================================

final_word_coverage_df.to_csv(
    AUDIT_ROOT
    / "final_word_coverage_capped300.csv",
    index=False,
    encoding="utf-8",
)


eligible_vocab_df.to_csv(
    AUDIT_ROOT
    / "eligible_vocabulary_min100.csv",
    index=False,
    encoding="utf-8",
)


final_threshold_df.to_csv(
    AUDIT_ROOT
    / "final_coverage_by_threshold.csv",
    index=False,
    encoding="utf-8",
)


final_target_coverage_df.to_csv(
    AUDIT_ROOT
    / "final_target_coverage.csv",
    index=False,
    encoding="utf-8",
)


eligible_source_df.to_csv(
    AUDIT_ROOT
    / "eligible_vocabulary_source_composition_cap300.csv",
    index=False,
    encoding="utf-8",
)


eligible_doc_split_df.to_csv(
    AUDIT_ROOT
    / "eligible_vocabulary_doc_split_composition_cap300.csv",
    index=False,
    encoding="utf-8",
)


save_json(
    final_summary,
    AUDIT_ROOT
    / "final_corpus_coverage_summary.json",
)


# ============================================================
# Mostrar
# ============================================================

print(
    "\n=== Cobertura final por mínimo ==="
)

display(
    final_threshold_df
)


print(
    "\n=== Targets finales ==="
)

display(
    final_target_coverage_df
)


print(
    "\n=== Fuentes del vocabulario elegible (cap 300) ==="
)

display(
    eligible_source_df
)


print(
    "\n=== doc_split del vocabulario elegible (cap 300) ==="
)

display(
    eligible_doc_split_df
)


print_copy_block(
    "COBERTURA CONSOLIDADA DEL CORPUS",
    final_summary,
)


=== Cobertura final por mínimo ===


,min_occurrences,n_words,fraction_of_valid_vocabulary
0,1,469330,1.000000
1,2,279127,0.594735
2,5,159067,0.338924
3,10,108721,0.231652
4,20,74965,0.159728
5,50,45717,0.097409
6,75,36220,0.077174
7,100,30572,0.065140
8,125,26768,0.057034
9,150,23971,0.051075



=== Targets finales ===


,word,raw_valid_occurrences_seen,n_usable_unique_contexts_capped,true_class,class_label,reaches_min100
0,apple,2627,300,0,apple,True
1,olive,942,300,0,apple,True
2,pear,175,173,0,apple,True
3,bicycle,780,300,1,bicycle,True
4,parachute,1812,300,1,bicycle,True
5,scooter,100,100,1,bicycle,True
6,bread,941,300,2,bread,True
7,cheese,812,300,2,bread,True
8,peanut,101,101,2,bread,True
9,bus,4315,300,3,canoe,True



=== Fuentes del vocabulario elegible (cap 300) ===


,source_name,n_contexts_for_eligible_vocabulary_cap300
0,wikitext_103_raw,7225707
1,wikimedia_wikipedia_20231101_en_fallback,219



=== doc_split del vocabulario elegible (cap 300) ===


,doc_split,n_contexts_for_eligible_vocabulary_cap300
0,test,1082818
1,train,5057753
2,validation,1085355



PARA COPIAR Y PEGAR — COBERTURA CONSOLIDADA DEL CORPUS
{
  "all_targets_ge_100": true,
  "audit_max_contexts_per_word": 300,
  "coverage_by_threshold": [
    {
      "fraction_of_valid_vocabulary": 1.0,
      "min_occurrences": 1,
      "n_words": 469330
    },
    {
      "fraction_of_valid_vocabulary": 0.5947350478341465,
      "min_occurrences": 2,
      "n_words": 279127
    },
    {
      "fraction_of_valid_vocabulary": 0.3389235719003686,
      "min_occurrences": 5,
      "n_words": 159067
    },
    {
      "fraction_of_valid_vocabulary": 0.23165150320669892,
      "min_occurrences": 10,
      "n_words": 108721
    },
    {
      "fraction_of_valid_vocabulary": 0.15972769692966568,
      "min_occurrences": 20,
      "n_words": 74965
    },
    {
      "fraction_of_valid_vocabulary": 0.09740907250761724,
      "min_occurrences": 50,
      "n_words": 45717
    },
    {
      "fraction_of_valid_vocabulary": 0.07717384356422986,
      "min_occurrences": 75,
      "n_words": 36220
 

In [12]:
# ============================================================
# 12. Proyección de tamaño del dataset regenerado
# ============================================================
#
# Para cada cap C:
#
#   filas(C) =
#       sum_{w in V100} min(n_w, C)
#
# donde V100 contiene solo palabras con >=100 contextos
# utilizables.
#
# Esto preserva la estructura occurrence-level:
# siguen siendo filas de ocurrencias, no prototipos.
# ============================================================


# ============================================================
# 1. Calibración empírica opcional con Parquet actual
# ============================================================

observed_bytes_per_row = None
current_parquet_rows = None
current_parquet_size_bytes = None


if (
    CURRENT_CONTEXTUAL_PARQUET.exists()
    and pq is not None
):

    try:

        current_pf = pq.ParquetFile(
            CURRENT_CONTEXTUAL_PARQUET
        )

        current_parquet_rows = int(
            current_pf.metadata.num_rows
        )

        current_parquet_size_bytes = int(
            CURRENT_CONTEXTUAL_PARQUET.stat().st_size
        )

        if current_parquet_rows > 0:

            observed_bytes_per_row = float(
                current_parquet_size_bytes
                / current_parquet_rows
            )

    except Exception as exc:

        print(
            "No se pudo calibrar tamaño con el Parquet actual:",
            repr(exc),
        )


# Tamaño mínimo teórico solo del vector float32[768].
embedding_payload_bytes_per_row = (
    768
    * 4
)


projection_rows = []


for cap in PROJECTION_CAPS:

    effective_counts = np.minimum(
        eligible_vocab_df[
            "n_usable_unique_contexts_capped"
        ].to_numpy(
            dtype=np.int64
        ),
        int(
            cap
        ),
    )


    projected_rows = int(
        effective_counts.sum()
    )


    is_target_array = (
        eligible_vocab_df[
            "is_target"
        ]
        .to_numpy(
            dtype=bool
        )
    )


    projected_target_rows = int(
        effective_counts[
            is_target_array
        ].sum()
    )


    projected_filler_rows = int(
        projected_rows
        - projected_target_rows
    )


    theoretical_embedding_payload_gib = float(
        projected_rows
        * embedding_payload_bytes_per_row
        / 1024**3
    )


    estimated_parquet_gib = (
        float(
            projected_rows
            * observed_bytes_per_row
            / 1024**3
        )
        if observed_bytes_per_row
        is not None
        else None
    )


    projection_rows.append(
        {
            "cap_per_word":
                int(
                    cap
                ),

            "n_eligible_words":
                int(
                    len(
                        eligible_vocab_df
                    )
                ),

            "projected_total_rows":
                projected_rows,

            "projected_target_rows":
                projected_target_rows,

            "projected_filler_rows":
                projected_filler_rows,

            "mean_rows_per_eligible_word":
                float(
                    projected_rows
                    / len(
                        eligible_vocab_df
                    )
                )
                if len(
                    eligible_vocab_df
                )
                else None,

            "embedding_payload_only_gib":
                theoretical_embedding_payload_gib,

            "estimated_full_parquet_gib_from_current_dataset":
                estimated_parquet_gib,
        }
    )


dataset_projection_df = pd.DataFrame(
    projection_rows
)


projection_summary = {
    "n_eligible_words":
        int(
            len(
                eligible_vocab_df
            )
        ),

    "selection_rule":
        "word_has_at_least_100_usable_unique_contexts",

    "projection_rule":
        "sum(min(n_usable_contexts, cap)) over eligible words",

    "embedding_payload_bytes_per_row_float32_768":
        int(
            embedding_payload_bytes_per_row
        ),

    "current_parquet_calibration": {
        "path":
            str(
                CURRENT_CONTEXTUAL_PARQUET
            ),

        "exists":
            bool(
                CURRENT_CONTEXTUAL_PARQUET.exists()
            ),

        "rows":
            current_parquet_rows,

        "size_bytes":
            current_parquet_size_bytes,

        "observed_bytes_per_row":
            observed_bytes_per_row,
    },

    "projections":
        dataset_projection_df,
}


dataset_projection_df.to_csv(
    AUDIT_ROOT
    / "dataset_size_projections.csv",
    index=False,
    encoding="utf-8",
)


save_json(
    projection_summary,
    AUDIT_ROOT
    / "dataset_size_projections.json",
)


display(
    dataset_projection_df
)


print_copy_block(
    "PROYECCIÓN DATASET REGENERADO",
    projection_summary,
)

,cap_per_word,n_eligible_words,projected_total_rows,projected_target_rows,projected_filler_rows,mean_rows_per_eligible_word,embedding_payload_only_gib,estimated_full_parquet_gib_from_current_dataset
0,100,30572,3057200,2400,3054800,100.000000,8.746719,11.316058
1,150,30572,4400031,3312,4396719,143.923558,12.588590,16.286473
2,200,30572,5488630,4098,5484532,179.531270,15.703096,20.315862
3,250,30572,6415571,4848,6410723,209.851204,18.355096,23.746883
4,300,30572,7225926,5587,7220339,236.357648,20.673540,26.746368



PARA COPIAR Y PEGAR — PROYECCIÓN DATASET REGENERADO
{
  "current_parquet_calibration": {
    "exists": true,
    "observed_bytes_per_row": 3974.396313333333,
    "path": "C:\\Users\\jisaa\\Workspace\\Tesis\\contextual_dataset_definitivo\\contextual_occurrences_clean_mincoverage.parquet",
    "rows": 300000,
    "size_bytes": 1192318894
  },
  "embedding_payload_bytes_per_row_float32_768": 3072,
  "n_eligible_words": 30572,
  "projection_rule": "sum(min(n_usable_contexts, cap)) over eligible words",
  "projections": [
    {
      "cap_per_word": 100,
      "embedding_payload_only_gib": 8.746719360351562,
      "estimated_full_parquet_gib_from_current_dataset": 11.316057675632337,
      "mean_rows_per_eligible_word": 100.0,
      "n_eligible_words": 30572,
      "projected_filler_rows": 3054800,
      "projected_target_rows": 2400,
      "projected_total_rows": 3057200
    },
    {
      "cap_per_word": 150,
      "embedding_payload_only_gib": 12.588589668273926,
      "estimated_full_p

In [13]:
# ============================================================
# 13. Resumen final mínimo para decisión de regeneración
# ============================================================
#
# Este bloque es el más compacto para compartir.
#
# No fija arbitrariamente cuántas palabras son "suficientes".
# Entrega la evidencia necesaria para decidir el diseño final:
#
# - V100 real del corpus;
# - cobertura de targets;
# - comparación descriptiva con 458 y GloVe;
# - tamaño resultante para cap 300;
# - fuentes disponibles/fallidas.
# ============================================================


projection_cap300_row = (
    dataset_projection_df.loc[
        dataset_projection_df[
            "cap_per_word"
        ]
        == 300
    ]
)


if len(
    projection_cap300_row
) == 1:

    cap300_record = (
        projection_cap300_row
        .iloc[0]
        .to_dict()
    )

else:

    cap300_record = None


decision_summary = {
    "n_eligible_words_ge_100":
        int(
            len(
                eligible_vocab_df
            )
        ),

    "previous_dataset_eligible_words_ge_100":
        int(
            CURRENT_CONTEXTUAL_V100_REFERENCE
        ),

    "gain_factor_vs_previous_dataset":
        safe_fraction(
            len(
                eligible_vocab_df
            ),
            CURRENT_CONTEXTUAL_V100_REFERENCE,
        ),

    "glove_reference_vocab_size":
        int(
            GLOVE_VOCAB_REFERENCE
        ),

    "eligible_fraction_of_glove_reference":
        safe_fraction(
            len(
                eligible_vocab_df
            ),
            GLOVE_VOCAB_REFERENCE,
        ),

    "all_24_targets_ge_100":
        bool(
            final_target_coverage_df[
                "reaches_min100"
            ].all()
        ),

    "targets_below_100":
        final_target_coverage_df.loc[
            ~final_target_coverage_df[
                "reaches_min100"
            ],
            "word",
        ].tolist(),

    "target_fallback_used":
        bool(
            target_fallback_report[
                "fallback_used"
            ]
        ),

    "target_fallback_error":
        target_fallback_report[
            "error"
        ],

    "n_general_source_failures":
        int(
            len(
                general_state[
                    "source_failures"
                ]
            )
        ),

    "general_source_failures":
        general_state[
            "source_failures"
        ],

    "cap300_projection":
        cap300_record,

    "results_root":
        str(
            AUDIT_ROOT
        ),
}


save_json(
    decision_summary,
    AUDIT_ROOT
    / "decision_summary.json",
)


print_copy_block(
    "RESUMEN FINAL CORPUS_AUDIT",
    decision_summary,
)


PARA COPIAR Y PEGAR — RESUMEN FINAL CORPUS_AUDIT
{
  "all_24_targets_ge_100": true,
  "cap300_projection": {
    "cap_per_word": 300.0,
    "embedding_payload_only_gib": 20.673540115356445,
    "estimated_full_parquet_gib_from_current_dataset": 26.746367714199682,
    "mean_rows_per_eligible_word": 236.3576475206071,
    "n_eligible_words": 30572.0,
    "projected_filler_rows": 7220339.0,
    "projected_target_rows": 5587.0,
    "projected_total_rows": 7225926.0
  },
  "eligible_fraction_of_glove_reference": 0.07643057322929922,
  "gain_factor_vs_previous_dataset": 66.75109170305677,
  "general_source_failures": [
    {
      "config": "20220301.en",
      "dataset_name": "wikipedia",
      "error": "RuntimeError('Dataset scripts are no longer supported, but found wikipedia.py')",
      "source_name": "wikipedia_20220301_en",
      "split": "train"
    },
    {
      "config": null,
      "dataset_name": "bookcorpusopen",
      "error": "RuntimeError('Dataset scripts are no longer sup

## Resultados que conviene entregar para revisión

Después de ejecutar el notebook completo, los bloques principales para copiar y pegar son:

1. **`AUDITORÍA GENERAL PRE-BERT`**
2. **`FALLBACK DIRIGIDO TARGETS`**
3. **`COBERTURA CONSOLIDADA DEL CORPUS`**
4. **`PROYECCIÓN DATASET REGENERADO`**
5. **`RESUMEN FINAL CORPUS_AUDIT`**

Con esos resultados se puede decidir la política definitiva de regeneración sin ejecutar BERT todavía.

### Artefactos guardados

Los CSV/JSON relevantes se guardan en:

`C:\Users\jisaa\Workspace\Tesis\Exp2\Results\ResultsExp2_prototipos\00_corpus_audit`

Entre ellos:

- `target_words_contract.csv`
- `occurrence_schema_contract.json`
- `general_word_coverage_capped300.csv`
- `general_coverage_by_threshold.csv`
- `general_target_coverage.csv`
- `general_source_split_stats.csv`
- `general_source_failures.json`
- `target_fallback_report.json`
- `final_word_coverage_capped300.csv`
- `eligible_vocabulary_min100.csv`
- `final_coverage_by_threshold.csv`
- `final_target_coverage.csv`
- `eligible_vocabulary_source_composition_cap300.csv`
- `eligible_vocabulary_doc_split_composition_cap300.csv`
- `dataset_size_projections.csv`
- `dataset_size_projections.json`
- `decision_summary.json`

El siguiente paso, una vez revisados estos resultados, será modificar el generador contextual para que **seleccione y conserve occurrence-level contexts para todo el vocabulario elegible**, y solo posteriormente ejecutar BERT.

In [14]:
# ============================================================
# 14. AUDITORÍA FINAL DEL POOL COMÚN
#     Wikitext-103 + Wikimedia/Wikipedia para TODAS las palabras
# ============================================================
#
# OBJETIVO
# --------
#
# Hasta ahora:
#
# - Wikitext-103 se utilizó como corpus general.
# - Wikimedia/Wikipedia 20231101 se utilizó únicamente como
#   fallback dirigido a palabras experimentales deficitarias.
#
# La decisión metodológica final cambia esa lógica:
#
#     TODAS las palabras
#            ↓
#     mismo pool de corpus
#            ↓
#     mismas reglas léxicas
#            ↓
#     mismas reglas de contexto
#            ↓
#     elegibles si >= 100 contextos únicos
#
# Por tanto, Wikimedia/Wikipedia se incorpora ahora como
# fuente GENERAL y no como fuente especial de CategoryWords.
#
# IMPORTANTE
# ----------
#
# Esta celda:
#
# - NO ejecuta BERT;
# - NO vuelve a recorrer Wikitext-103;
# - reutiliza general_state;
# - mantiene cap de auditoría = 300;
# - deduplica globalmente por (word, context_window).
#
# Si previamente se ejecutó el fallback dirigido, los contextos
# obtenidos allí proceden de la misma fuente ahora permitida.
# No se duplicarán gracias a seen_pair_hashes.
# ============================================================


COMMON_WIKIPEDIA_SOURCE_NAME = (
    "wikimedia_wikipedia_20231101_en_common"
)

COMMON_WIKIPEDIA_DATASET_NAME = (
    "wikimedia/wikipedia"
)

COMMON_WIKIPEDIA_CONFIG = (
    "20231101.en"
)

COMMON_WIKIPEDIA_SPLIT = (
    "train"
)

# Mantener el mismo orden de magnitud que la antigua fuente
# Wikipedia general del generador contextual.
COMMON_WIKIPEDIA_MAX_RECORDS = 800_000


# ============================================================
# 1. Estado antes de ampliar el pool
# ============================================================

v100_before_common_wikipedia = int(
    sum(
        count >= MIN_OCCURRENCES_FOR_PROTOTYPE
        for count
        in general_state[
            "usable_context_counts"
        ].values()
    )
)


n_unique_contexts_before = int(
    len(
        general_state[
            "seen_pair_hashes"
        ]
    )
)


print(
    "V100 antes de Wikipedia común:",
    v100_before_common_wikipedia,
)

print(
    "Contextos únicos antes:",
    n_unique_contexts_before,
)


# ============================================================
# 2. Incorporar Wikipedia como fuente general
# ============================================================

scan_one_hf_source_split(
    state=general_state,

    source_name=(
        COMMON_WIKIPEDIA_SOURCE_NAME
    ),

    dataset_name=(
        COMMON_WIKIPEDIA_DATASET_NAME
    ),

    config=(
        COMMON_WIKIPEDIA_CONFIG
    ),

    split=(
        COMMON_WIKIPEDIA_SPLIT
    ),

    text_field="text",

    streaming=True,

    max_records_per_split=(
        COMMON_WIKIPEDIA_MAX_RECORDS
    ),
)


# ============================================================
# 3. Reconstruir cobertura final del pool común
# ============================================================

common_pool_word_coverage_df = (
    build_word_coverage_dataframe(
        general_state
    )
)


common_pool_threshold_df = (
    build_threshold_dataframe(
        common_pool_word_coverage_df
    )
)


common_pool_eligible_vocab_df = (
    common_pool_word_coverage_df.loc[
        common_pool_word_coverage_df[
            "n_usable_unique_contexts_capped"
        ]
        >= MIN_OCCURRENCES_FOR_PROTOTYPE
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


common_pool_v100 = int(
    len(
        common_pool_eligible_vocab_df
    )
)


# ============================================================
# 4. Cobertura de las 8 CategoryWords
# ============================================================

category_coverage_rows = []


for category_word in category_words:

    row = (
        common_pool_word_coverage_df.loc[
            common_pool_word_coverage_df[
                "word"
            ]
            == category_word
        ]
    )


    if row.empty:

        n_raw = 0
        n_contexts = 0

    else:

        row = row.iloc[0]

        n_raw = int(
            row[
                "raw_valid_occurrences_seen"
            ]
        )

        n_contexts = int(
            row[
                "n_usable_unique_contexts_capped"
            ]
        )


    category_coverage_rows.append(
        {
            "category_word":
                category_word,

            "raw_valid_occurrences_seen":
                n_raw,

            "n_usable_unique_contexts_capped":
                n_contexts,

            "reaches_100":
                bool(
                    n_contexts
                    >= MIN_OCCURRENCES_FOR_PROTOTYPE
                ),

            "reaches_300":
                bool(
                    n_contexts
                    >= AUDIT_MAX_CONTEXTS_PER_WORD
                ),
        }
    )


common_pool_category_coverage_df = (
    pd.DataFrame(
        category_coverage_rows
    )
)


all_category_words_ge100 = bool(
    common_pool_category_coverage_df[
        "reaches_100"
    ].all()
)


# ============================================================
# 5. Estadísticos del pool definitivo
# ============================================================

n_unique_contexts_after = int(
    len(
        general_state[
            "seen_pair_hashes"
        ]
    )
)


common_pool_summary = {

    "pool_definition":
        (
            "wikitext_103_raw + "
            "wikimedia_wikipedia_20231101_en"
        ),

    "same_sources_available_to_all_words":
        True,

    "special_target_sampling":
        False,

    "min_occurrences_per_word":
        int(
            MIN_OCCURRENCES_FOR_PROTOTYPE
        ),

    "planned_final_occurrences_per_word":
        100,

    "audit_cap_per_word":
        int(
            AUDIT_MAX_CONTEXTS_PER_WORD
        ),

    "wikipedia_common_max_records":
        int(
            COMMON_WIKIPEDIA_MAX_RECORDS
        ),

    "n_valid_unique_words_observed":
        int(
            len(
                common_pool_word_coverage_df
            )
        ),

    "n_eligible_words_ge100":
        common_pool_v100,

    "n_eligible_words_before_common_wikipedia":
        int(
            v100_before_common_wikipedia
        ),

    "gain_in_v100_from_common_wikipedia":
        int(
            common_pool_v100
            - v100_before_common_wikipedia
        ),

    "gain_factor_vs_old_contextual_dataset_458":
        float(
            common_pool_v100
            / 458
        ),

    "fraction_of_glove_reference":
        float(
            common_pool_v100
            / GLOVE_VOCAB_REFERENCE
        ),

    "n_unique_contexts_before_common_wikipedia":
        n_unique_contexts_before,

    "n_unique_contexts_after_common_wikipedia":
        n_unique_contexts_after,

    "additional_unique_contexts":
        int(
            n_unique_contexts_after
            - n_unique_contexts_before
        ),

    "all_8_category_words_ge100":
        all_category_words_ge100,

    "category_words":
        common_pool_category_coverage_df,

    "coverage_by_threshold":
        common_pool_threshold_df,
}


# ============================================================
# 6. Validación metodológica fuerte
# ============================================================

if not all_category_words_ge100:

    missing_category_words = (
        common_pool_category_coverage_df.loc[
            ~common_pool_category_coverage_df[
                "reaches_100"
            ],
            "category_word",
        ]
        .tolist()
    )

    raise RuntimeError(
        "El pool común no permite obtener 100 contextos "
        "para todas las CategoryWords. "
        f"Problemáticas: {missing_category_words}"
    )


# ============================================================
# 7. Guardar artefactos
# ============================================================

common_pool_word_coverage_df.to_csv(
    AUDIT_ROOT
    / "common_pool_word_coverage_capped300.csv",
    index=False,
    encoding="utf-8",
)


common_pool_eligible_vocab_df.to_csv(
    AUDIT_ROOT
    / "common_pool_eligible_vocabulary_min100.csv",
    index=False,
    encoding="utf-8",
)


common_pool_threshold_df.to_csv(
    AUDIT_ROOT
    / "common_pool_coverage_by_threshold.csv",
    index=False,
    encoding="utf-8",
)


common_pool_category_coverage_df.to_csv(
    AUDIT_ROOT
    / "common_pool_category_words_coverage.csv",
    index=False,
    encoding="utf-8",
)


save_json(
    common_pool_summary,
    AUDIT_ROOT
    / "common_pool_summary.json",
)


# ============================================================
# 8. Mostrar
# ============================================================

print(
    "\n=== Cobertura por umbral — POOL COMÚN ==="
)

display(
    common_pool_threshold_df
)


print(
    "\n=== Cobertura de las 8 CategoryWords ==="
)

display(
    common_pool_category_coverage_df
)


print(
    "\nV100 final del pool común:",
    common_pool_v100,
)


print_copy_block(
    "POOL COMÚN WIKITEXT + WIKIPEDIA",
    common_pool_summary,
)

V100 antes de Wikipedia común: 30572
Contextos únicos antes: 10499876

AUDITANDO: wikimedia_wikipedia_20231101_en_common | split= train | streaming= True | max_records= 800000


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

wikimedia_wikipedia_20231101_en_common/train:   0%|          | 0/800000 [00:00<?, ?docs/s]


Resumen source/split:
{
  "source_name": "wikimedia_wikipedia_20231101_en_common",
  "split": "train",
  "dataset_name": "wikimedia/wikipedia",
  "config": "20231101.en",
  "streaming": true,
  "max_records_per_split": 800000,
  "elapsed_seconds": 980.5176431,
  "documents_seen": 800000,
  "tokens_seen": 456043316,
  "documents_usable": 799879,
  "valid_word_tokens": 259377714,
  "unique_contexts_accepted": 28629066,
  "documents_contributing": 773222,
  "too_short": 121
}

=== Cobertura por umbral — POOL COMÚN ===


,min_occurrences,n_words,fraction_of_valid_vocabulary
0,1,2533074,1.000000
1,2,1331782,0.525757
2,5,707791,0.279420
3,10,460583,0.181828
4,20,295409,0.116621
5,50,161536,0.063771
6,75,123941,0.048929
7,100,103033,0.040675
8,125,89365,0.035279
9,150,79653,0.031445



=== Cobertura de las 8 CategoryWords ===


,category_word,raw_valid_occurrences_seen,n_usable_unique_contexts_capped,reaches_100,reaches_300
0,apple,16359,300,True,True
1,bicycle,8645,300,True,True
2,bread,7289,300,True,True
3,canoe,3591,300,True,True
4,chair,24547,300,True,True
5,jellyfish,826,300,True,True
6,screwdriver,290,251,True,False
7,skunk,812,300,True,True



V100 final del pool común: 103033

PARA COPIAR Y PEGAR — POOL COMÚN WIKITEXT + WIKIPEDIA
{
  "additional_unique_contexts": 28629066,
  "all_8_category_words_ge100": true,
  "audit_cap_per_word": 300,
  "category_words": [
    {
      "category_word": "apple",
      "n_usable_unique_contexts_capped": 300,
      "raw_valid_occurrences_seen": 16359,
      "reaches_100": true,
      "reaches_300": true
    },
    {
      "category_word": "bicycle",
      "n_usable_unique_contexts_capped": 300,
      "raw_valid_occurrences_seen": 8645,
      "reaches_100": true,
      "reaches_300": true
    },
    {
      "category_word": "bread",
      "n_usable_unique_contexts_capped": 300,
      "raw_valid_occurrences_seen": 7289,
      "reaches_100": true,
      "reaches_300": true
    },
    {
      "category_word": "canoe",
      "n_usable_unique_contexts_capped": 300,
      "raw_valid_occurrences_seen": 3591,
      "reaches_100": true,
      "reaches_300": true
    },
    {
      "category_word": "

In [15]:
# ============================================================
# 15. AUDITORÍA LÉXICA DEL VOCABULARIO FINAL V100
# ============================================================
#
# OBJETIVO
# --------
#
# Verificar que las palabras candidatas:
#
# - sean tokens alfabéticos ingleses;
# - tengan longitud >= 3;
# - no sean stopwords;
# - no sean CUSTOM_BAD_WORDS;
# - no contengan basura evidente;
#
# Además:
#
# - mostrar las 100 palabras más frecuentes;
# - generar una muestra aleatoria reproducible de 500 palabras;
# - resumir la longitud de los tokens.
#
# IMPORTANTE:
#
# Esta celda es DIAGNÓSTICA.
#
# No se aplica POS filtering.
# No se restringe a sustantivos.
# No se eliminan palabras adicionales automáticamente.
# ============================================================


LEXICAL_AUDIT_RANDOM_SEED = 42
LEXICAL_AUDIT_SAMPLE_SIZE = 500
LEXICAL_AUDIT_TOP_N = 100


lexical_df = (
    common_pool_eligible_vocab_df
    .copy()
)


# ============================================================
# 1. Propiedades léxicas
# ============================================================

lexical_df[
    "word_length"
] = (
    lexical_df[
        "word"
    ]
    .astype(str)
    .str.len()
)


lexical_df[
    "is_strict_alpha"
] = (
    lexical_df[
        "word"
    ]
    .astype(str)
    .str.fullmatch(
        r"[a-z]+"
    )
    .fillna(False)
)


lexical_df[
    "is_stopword"
] = (
    lexical_df[
        "word"
    ]
    .isin(
        STOPWORDS
    )
)


lexical_df[
    "is_custom_bad_word"
] = (
    lexical_df[
        "word"
    ]
    .isin(
        CUSTOM_BAD_WORDS
    )
)


lexical_df[
    "is_bad_word_union"
] = (
    lexical_df[
        "word"
    ]
    .isin(
        BAD_WORDS
    )
)


lexical_df[
    "too_short"
] = (
    lexical_df[
        "word_length"
    ]
    < MIN_ROW_WORD_LEN
)


lexical_df[
    "passes_lexical_contract"
] = (
    lexical_df[
        "is_strict_alpha"
    ]
    &
    ~lexical_df[
        "is_stopword"
    ]
    &
    ~lexical_df[
        "is_custom_bad_word"
    ]
    &
    ~lexical_df[
        "too_short"
    ]
)


# ============================================================
# 2. Violaciones
# ============================================================

lexical_violations_df = (
    lexical_df.loc[
        ~lexical_df[
            "passes_lexical_contract"
        ]
    ]
    .copy()
    .sort_values(
        [
            "raw_valid_occurrences_seen",
            "word",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 3. Palabras más frecuentes
# ============================================================

top_frequency_words_df = (
    lexical_df
    .sort_values(
        [
            "raw_valid_occurrences_seen",
            "word",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .head(
        LEXICAL_AUDIT_TOP_N
    )
    [
        [
            "word",
            "raw_valid_occurrences_seen",
            "n_usable_unique_contexts_capped",
            "word_length",
        ]
    ]
    .reset_index(
        drop=True
    )
)


# ============================================================
# 4. Muestra aleatoria reproducible
# ============================================================

sample_size = min(
    LEXICAL_AUDIT_SAMPLE_SIZE,
    len(
        lexical_df
    ),
)


random_vocabulary_sample_df = (
    lexical_df.sample(
        n=sample_size,
        random_state=(
            LEXICAL_AUDIT_RANDOM_SEED
        ),
        replace=False,
    )
    [
        [
            "word",
            "raw_valid_occurrences_seen",
            "n_usable_unique_contexts_capped",
            "word_length",
        ]
    ]
    .sort_values(
        "word"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 5. Distribución de longitudes
# ============================================================

word_length_distribution = (
    lexical_df[
        "word_length"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .to_dict()
)


# ============================================================
# 6. Resumen
# ============================================================

lexical_audit_summary = {

    "n_v100_words":
        int(
            len(
                lexical_df
            )
        ),

    "lexical_contract": {
        "regex":
            "[a-z]+",

        "minimum_word_length":
            int(
                MIN_ROW_WORD_LEN
            ),

        "exclude_english_stopwords":
            True,

        "exclude_custom_bad_words":
            True,

        "pos_filtering":
            False,

        "noun_only":
            False,
    },

    "n_non_alpha":
        int(
            (
                ~lexical_df[
                    "is_strict_alpha"
                ]
            ).sum()
        ),

    "n_stopwords":
        int(
            lexical_df[
                "is_stopword"
            ].sum()
        ),

    "n_custom_bad_words":
        int(
            lexical_df[
                "is_custom_bad_word"
            ].sum()
        ),

    "n_bad_words_union":
        int(
            lexical_df[
                "is_bad_word_union"
            ].sum()
        ),

    "n_too_short":
        int(
            lexical_df[
                "too_short"
            ].sum()
        ),

    "n_lexical_contract_violations":
        int(
            len(
                lexical_violations_df
            )
        ),

    "all_words_pass_lexical_contract":
        bool(
            lexical_df[
                "passes_lexical_contract"
            ].all()
        ),

    "word_length_distribution":
        word_length_distribution,

    "top_100_words_by_frequency":
        top_frequency_words_df[
            "word"
        ].tolist(),

    # El CSV guarda 500.
    # El bloque para copiar incluye solo 100
    # para no producir una salida excesiva.
    "random_sample_100_words":
        random_vocabulary_sample_df[
            "word"
        ]
        .head(
            100
        )
        .tolist(),
}


# ============================================================
# 7. Validaciones
# ============================================================

if len(
    lexical_violations_df
) > 0:

    print(
        "\nADVERTENCIA:"
    )

    print(
        "Se encontraron palabras que violan "
        "el contrato léxico."
    )

    display(
        lexical_violations_df.head(
            100
        )
    )

else:

    print(
        "\nOK: ninguna palabra de V100 viola "
        "el contrato léxico automático."
    )


# ============================================================
# 8. Guardar artefactos
# ============================================================

lexical_df.to_csv(
    AUDIT_ROOT
    / "v100_lexical_audit_full.csv",
    index=False,
    encoding="utf-8",
)


lexical_violations_df.to_csv(
    AUDIT_ROOT
    / "v100_lexical_contract_violations.csv",
    index=False,
    encoding="utf-8",
)


top_frequency_words_df.to_csv(
    AUDIT_ROOT
    / "v100_top100_frequency_words.csv",
    index=False,
    encoding="utf-8",
)


random_vocabulary_sample_df.to_csv(
    AUDIT_ROOT
    / "v100_random_sample_500.csv",
    index=False,
    encoding="utf-8",
)


save_json(
    lexical_audit_summary,
    AUDIT_ROOT
    / "v100_lexical_audit_summary.json",
)


# ============================================================
# 9. Mostrar
# ============================================================

print(
    "\n=== TOP 100 PALABRAS POR FRECUENCIA ==="
)

display(
    top_frequency_words_df
)


print(
    "\n=== MUESTRA ALEATORIA DE 500 PALABRAS ==="
)

display(
    random_vocabulary_sample_df
)


print(
    "\n=== DISTRIBUCIÓN DE LONGITUD ==="
)

display(
    pd.DataFrame(
        [word_length_distribution]
    )
    .T
    .rename(
        columns={
            0: "value"
        }
    )
)


print_copy_block(
    "AUDITORÍA LÉXICA V100",
    lexical_audit_summary,
)


OK: ninguna palabra de V100 viola el contrato léxico automático.

=== TOP 100 PALABRAS POR FRECUENCIA ===


,word,raw_valid_occurrences_seen,n_usable_unique_contexts_capped,word_length
0,new,1175722,300,3
1,american,788629,300,8
2,time,702162,300,4
3,people,673872,300,6
4,references,672026,300,10
...,...,...,...,...
95,place,242687,300,5
96,like,242292,300,4
97,television,241695,300,10
98,town,241655,300,4



=== MUESTRA ALEATORIA DE 500 PALABRAS ===


,word,raw_valid_occurrences_seen,n_usable_unique_contexts_capped,word_length
0,abir,184,184,4
1,abo,423,300,3
2,acadians,466,300,8
3,accredits,103,103,9
4,adak,171,165,4
...,...,...,...,...
495,xing,770,300,4
496,zahra,386,300,5
497,zest,319,300,4
498,zora,477,300,4



=== DISTRIBUCIÓN DE LONGITUD ===


,value
count,103033.000000
mean,7.213165
std,2.512905
min,3.000000
1%,3.000000
5%,3.000000
10%,4.000000
25%,5.000000
50%,7.000000
75%,9.000000



PARA COPIAR Y PEGAR — AUDITORÍA LÉXICA V100
{
  "all_words_pass_lexical_contract": true,
  "lexical_contract": {
    "exclude_custom_bad_words": true,
    "exclude_english_stopwords": true,
    "minimum_word_length": 3,
    "noun_only": false,
    "pos_filtering": false,
    "regex": "[a-z]+"
  },
  "n_bad_words_union": 0,
  "n_custom_bad_words": 0,
  "n_lexical_contract_violations": 0,
  "n_non_alpha": 0,
  "n_stopwords": 0,
  "n_too_short": 0,
  "n_v100_words": 103033,
  "random_sample_100_words": [
    "abir",
    "abo",
    "acadians",
    "accredits",
    "adak",
    "administered",
    "adv",
    "advantageous",
    "agents",
    "aintree",
    "aldersgate",
    "alea",
    "amana",
    "amasis",
    "ambedkar",
    "amedeo",
    "anglin",
    "anka",
    "anonymus",
    "antiaircraft",
    "antone",
    "aoun",
    "arcelormittal",
    "archrival",
    "ark",
    "assun",
    "aurealis",
    "autocad",
    "bambaataa",
    "barrow",
    "battles",
    "bayard",
    "beamer",
  